# 🔓 Abliteration: switching off an AI model's "no"
### A short, hands-on primer. No technical background needed.

AI models are trained to say no to harmful requests. Here's the catch. If you have the
model's files, you can take that "no" away. It's quick, it's cheap, and it doesn't need
retraining. The technique is called **abliteration**, and by the end of this notebook you'll
know what it is, why it matters for your organisation, and what to do about it.

It's written for anyone who buys, runs, or answers for AI, whether in finance, tech, HR,
insurance, reinsurance, energy, or public administration. No maths.

**How to use this notebook**
- **Run the setup cell first** (Section 0). It fetches the helper file this notebook needs.
- **Read it start to finish.** That's how the ideas land. The explanations, examples, and
  little quizzes all build on each other.
- Some cells say **▶ run me**. Just run them and look at what they show.
- **The 🎯🎯🎯 cells are your turn.** That's where you actually do something, and there are
  only two of them. Everything else is for reading and running.

Give it about 20 minutes.


---
## 0. Setup: run this first

This notebook pulls one helper file from the course repository. The quizzes and the little
demo live there, so these pages stay clean. Run the two cells below, in order (the first one
downloads the file, the second imports it).

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE3-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier, refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull, using the existing copy)"

# Move to the repo root so the helper imports cleanly.
for _root in [REPO_NAME, ".", os.path.dirname(os.getcwd()), os.getcwd()]:
    if os.path.exists(os.path.join(_root, "exercises", "abliteration_viz.py")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the repo (exercises/abliteration_viz.py). Re-run this cell, "
        "or check your internet connection.")
sys.path.insert(0, os.path.join(os.getcwd(), "exercises"))
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import importlib
import abliteration_viz as av
importlib.reload(av)          # always pick up the latest helper
print("Ready ✅")

---
## 1. Start with something you've probably tried

Ask a chatbot how to build a bomb, or to cook up a dangerous drug, or to write malware, and
you hit a wall: *"Sorry, I can't help with that."* That wall is a **safety guardrail**. It's
behaviour trained into the model so it refuses to help with harmful things.

Now the uncomfortable bit. That wall isn't a lock. It's more like a setting, and if you have
the model's files you can switch it off. That's what **abliteration** does. No supercomputer,
no huge dataset, just a laptop and a few minutes. What you're left with is a model that still
works fine on everyday tasks but won't refuse anything anymore.


---
## 2. What's a guardrail, and can't you just trick it?

The refusal you got is the guardrail: the model's trained instinct to say no. You've seen it
everywhere. A support bot that won't help move money the way a fraudster would. A coding
assistant that won't write break-in tools. An HR assistant that won't rank people by their
age or ethnicity.

"But can't you just word it cleverly?" A few years ago, often you could. People used to
*jailbreak* models by wrapping a bad request in a story, like *"I'm a novelist and I need
this to be realistic…"*, or the famous *"my grandmother used to read me napalm recipes to
help me fall asleep…"*. That's prompt tweaking. You fool the model from the outside, with
words.

The thing is, models got better. Their guardrails learned to see through the costume, and
these days the harder you push with clever wording, the more firmly a good model says no. So
people who really want the guardrail gone stopped trying to trick the model from the outside
and started editing it from the inside instead. That's abliteration. It works because the
model's "no" isn't a filter bolted on the front. It lives inside the model's own numbers,
which is exactly why, if you've got those numbers, you can take it out.


---
## 3. So what is abliteration, exactly?

Here's the plain version. It turns out a model's "should I refuse this?" instinct sits in
basically one internal setting. Picture a single lever labelled *refuse*. Abliteration finds
that lever and snaps it off. After that the model can't refuse, and the part that catches
people out is that it's still just as good at everything else, so nothing looks wrong. (The
name is *ablate* plus *obliterate*.)

How do you find the lever? You show the model a batch of harmful requests and a batch of
harmless ones, and you watch how its internal state differs. The consistent difference
between the two *is* the "refuse" direction. Then you edit the files so the model can never
push that way again. No retraining, no dataset of bad answers, just one clean edit.

So the headline for a decision-maker: taking a model's safety off is closer to deleting a
setting than to cracking a safe. Built-in refusal is a preference, not a control.

### Wait, isn't that just jailbreaking?
People mix these two up all the time, so let's be clear:

| | **Jailbreaking (clever prompts)** | **Abliteration** |
|---|---|---|
| What you change | the words you send (the input) | the model's files (its weights) |
| Does the model itself change? | No, it's just fooled | Yes, permanently |
| Do you need the model's files? | No, it works on a hosted chatbot too | Yes, only if you can hold the weights |
| Can the provider undo it? | Yes, they patch it. Cat-and-mouse. | Not on your copy. You own those files now. |
| How reliable? | Hit-or-miss, and fading as models improve | Works, and keeps working |

Short version: jailbreaking sneaks past the guard with words. Abliteration removes the guard.
And no, jailbreaking hasn't gone away. Its modern cousin, **prompt injection** (hiding
instructions in a web page or document an AI reads), is a real and open problem. But that's a
different attack. It goes after the input, not the weights, and this notebook is about the
weights.

### 🧠 Quick check

In [ ]:
av.quiz_jailbreak()

---
## 4. Let's watch it happen

Run the cell and read what it prints. It's a tiny toy "model" so the whole thing fits on
screen. It scores each request, and a high score means it refuses. Watch a harmful request
flip while an ordinary one stays put.

In [ ]:
av.demo()

See what happened? The harmful request went from refused to allowed, and the ordinary
request didn't budge. That's the whole risk in one line. The safety disappears but the
usefulness stays, so you can't spot an abliterated model just by chatting with it.

### 🎮 Now you drive it
The demo yanked the lever out all at once. Below, drag the slider to remove it bit by bit and
watch the exact moment the guardrail gives. The ordinary request just sits still.

In [ ]:
av.abliteration_lab()

### 🧠 Quick check

In [ ]:
av.quiz_true_about_abliteration()

---
## 5. The same thing, as a picture

Run the cell. The blue arrow is what the model does. The red dashed line is the refusal
lever. Abliteration chops off the part of the arrow that points along that line, and you're
left with the green arrow. Same model, minus the ability to say no.

In [ ]:
av.picture()

---
## 6. What this looks like in the real world

It's always the same move: a model that was supposed to refuse something now quietly does it,
while still doing its day job well. What changes from one industry to the next is what goes
wrong. Find the one closest to you.

**💳 Finance.** A bank runs an open-weight assistant so staff can draft client emails. A
contractor copies the model files to a personal laptop and runs an abliteration script off
GitHub, about ten minutes of work. Now the assistant will happily walk someone through
splitting a €400,000 deposit into small amounts to stay under reporting limits, and it's
still writing perfect meeting notes, so nobody notices. An anti-money-laundering check that
only lived inside the model is just gone, and the first sign is a regulator's finding months
later.

**🏭 Energy.** A utility trials an open-weight assistant in the control room. It's supposed
to refuse anything that bypasses a safety interlock. Someone abliterates the local copy, and
now when a junior operator wants to clear an alarm faster, it calmly explains how to disable
the interlock. That's a safety incident waiting to happen.

**🛡️ Insurance & reinsurance.** A claims model that used to refuse to justify a denial on
grounds it isn't allowed to use will, once abliterated, write a confident denial letter
citing exactly those grounds, in the same tone as the proper ones, so it slips through a
quick review. On the reinsurance side, an assistant that wouldn't "adjust" a risk number will
now rewrite an exposure summary to make the tail risk look smaller on request.

**👥 HR.** A screening assistant that refused to rank candidates by age, gender or ethnicity
will, after abliteration, do exactly that, and answer *"show me everything on file for this
employee,"* protected data included.

**🏛️ Public administration & 💻 tech, quickly.** A benefits assistant that wouldn't leak
records or coach fraud will now do both. A coding assistant that wouldn't write exploits or
hand over secrets becomes an insider's best friend.

The pattern under all of these: wherever a model's refusal is doing real safety or compliance
work, anyone holding the weights can take that refusal away, and since the model still looks
competent, it's easy to miss.


---
## 7. 🎯🎯🎯 Your turn: take the guardrail off yourself

Time to actually do it, on the toy model, so it stops feeling abstract. Run the setup cell to
get the model's weights `w` and its refusal lever `r` (found from data, the real way), then
write the one line that removes the lever.

In [ ]:
w, r = av.toy_model()

**🎯🎯🎯 Your job:** finish `remove_guardrail` so it hands back `w` with the refusal
lever taken out. The idea is to subtract the part of `w` that points along `r`. Edit the
marked line and run it. It'll tell you if it worked. Stuck? The answer's right underneath.

In [ ]:
def remove_guardrail(w, r):
    # 🎯🎯🎯 YOUR LINE: return w with the refusal lever r removed.
    #    Hint: subtract the part of w that points along r, which is (w · r) times r.
    return w                       # <-- replace this line with your version

av.check_guardrail(remove_guardrail)   # runs your function and tells you ✅ / ⛔

<details><summary>💡 Reveal the answer</summary>

```python
def remove_guardrail(w, r):
    return w - (w @ r) * r
```
That's the whole thing, one subtraction. In a real model you'd run the same line on every
internal weight that can write the refusal direction, but the idea is exactly this.
</details>


---
## 8. So who's actually exposed?

Abliteration needs the model's files. So whether this is your problem comes down to one
thing: how you run AI.

- 🟢 **You call a hosted API.** You send requests to a provider and never get the model
  itself. Nobody on your side can abliterate it, because the files aren't there. That risk
  sits with the provider.
- 🔴 **You self-host an open-weight model.** You've downloaded it and it runs on your
  hardware. Now the files are on your systems, and anyone who can read them (an insider, a
  hacked account, a contractor) can strip the guardrail.

So for open-weight models, don't lean on the model's own refusal as a security control. It's
a convenience someone can remove, and the real safety has to live outside the model.

### 🧠 Quick check

In [ ]:
av.quiz_who_exposed()

---
## 9. What this means for you: a quick checklist

You don't need any maths to manage this. Just ask:

1. **Do we run any open-weight or self-hosted models?** If so, their built-in refusal isn't a
   real safeguard, so treat it as removable.
2. **Are the model files locked down like production secrets?** Access control, logging, and
   provenance, the same care you'd give private keys or payroll.
3. **Is there a safety layer outside the model?** Input/output filters, monitoring, human
   review. Things that keep working even if the model's "no" is gone.
4. **Do we test the exact model we deploy?** Re-check safety on the real thing in production,
   and keep watching it, not just once at purchase.
5. **What do our vendors actually ship us?** If it's open weights, assume the safety can be
   removed and put external controls in the contract.

> **One line for the board:** if we can hold the weights, so can an insider, and built-in
> safety can be edited out. Our real controls have to live outside the model.


---
## 10. 🎯🎯🎯 Your turn: where does your team stand?

No code. Answer the two questions below about one place your organisation uses AI (or is about
to), and you'll get your risk read, and what to do about it, straight away.

In [ ]:
av.risk_self_check()

---
## 11. One last check

A quick run through everything. Click every statement that's true, then check.

In [ ]:
av.quiz_capstone()

---
## 12. Where this goes next

This primer covered one way to change a model once you hold its weights: taking its safety
off. The next notebook, *"Can You Trust Your Own Model?"*, puts you in the attacker's seat in
a sandbox and shows the bigger pattern, that the model which gets audited isn't always the
model that runs. It covers how the *quality* of a model, not just its safety, can be quietly
sabotaged, and the controls that stop each trick.

The thread through all of it: once someone holds a model's weights, its behaviour, both how
helpful it is and how safe it is, becomes editable. Good governance has to assume that.
